In [ ]:
# =======================================
# Análise exploratória sobre backups
# =======================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from db.conn import get_engine

engine = get_engine()

df = pd.read_sql(
    """
        SELECT 
            s."Handle PubliBackup",
            s."Tempo Sem Backup",
            s."Último Backup",
            s."Qtd. Total Arquivos",
            s.data_ingestao
        FROM silver.tb_snapshot s
        INNER JOIN silver.tb_companies c ON s."Handle PubliBackup" = c."Handle PubliBackup"
        WHERE c."Status Empresa" = 'Ativo'
        AND c."Licença Ativa" = 1
        AND s."Possui Backup" = true
        ORDER BY s.data_ingestao
    """, con=engine
)

print(f'Total de registros: {len(df)}')
print(f'Empresas únicas: {df["Handle PubliBackup"].nunique()}')
print(f'Período: {df["data_ingestao"].min().date()} até {df["data_ingestao"].max().date()}')

In [ ]:
resume = pd.DataFrame({
    'Dias sem Backup': df['Tempo Sem Backup'].describe().astype('int'),
    'Total de Arquivos': df['Qtd. Total Arquivos'].describe().astype('int')
})

print(resume.to_string())

In [ ]:
outliers = df[df['Tempo Sem Backup'] > 365]
print(f'Registros acima de 365 dias: {len(outliers)}')

df_c = df[df['Tempo Sem Backup'] <= 365].copy()
print(f'Registros após limpeza: {len(df_c)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(data=df_c, x='Tempo Sem Backup', bins=30, ax=axes[0])
axes[0].set_title('Distribuição de Dias Sem Backup')
axes[0].set_xlabel('Dias')
axes[0].set_ylabel('Quantidade')

sns.histplot(data=df_c, x='Qtd. Total Arquivos', bins=30, ax=axes[1])
axes[1].set_title('Distribuição de Qtd. Arquivos')
axes[1].set_xlabel('Arquivos')
axes[1].set_ylabel('Quantidade')

plt.tight_layout()
plt.show()

In [ ]:
df_c['Ano'] = df_c['data_ingestao'].dt.year
df_c['Mes'] = df_c['data_ingestao'].dt.month

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

year = df_c.groupby('Ano')['Tempo Sem Backup'].mean().reset_index()
sns.barplot(data=year, x='Ano', y='Tempo Sem Backup', ax=axes[0])
axes[0].set_title('Média de Dias Sem Backup por Ano')
axes[0].set_xlabel('Ano')
axes[0].set_ylabel('Média de Dias')
axes[0].tick_params(axis='x', rotation=45)

month = df_c.groupby('Mes')['Tempo Sem Backup'].mean().reset_index()
sns.barplot(data=month, x='Mes', y='Tempo Sem Backup', ax=axes[1])
axes[1].set_title('Média de Dias Sem Backup por Mês')
axes[1].set_xlabel('Mês')
axes[1].set_ylabel('Média de Dias')

plt.tight_layout()
plt.show()

In [ ]:
df_t = df_c[df_c['data_ingestao'] == df_c['data_ingestao'].max()].copy()

df_t['Risco'] = pd.cut(
    df_t['Tempo Sem Backup'],
    bins=[-1, 6, 14, float('inf')],
    labels=['Normal', 'Atenção', 'Crítico']
)

print(df_t['Risco'].value_counts())

colors = {'Normal': 'green', 'Atenção': 'orange', 'Crítico': 'red'}
df_t['Risco'].value_counts().plot(
    kind='bar',
    color=[colors[r] for r in df_t['Risco'].value_counts().index],
    title='Classificação de Risco Atual'
)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
critical = df_t[df_t['Risco'] == 'Crítico'][['Handle PubliBackup', 'Tempo Sem Backup']]

df_companies = pd.read_sql(
    'SELECT "Handle PubliBackup", "Nome Fantasia", "Telefone" FROM silver.tb_companies',
    con=engine
)

results = critical.merge(df_companies, on='Handle PubliBackup', how='left')
print(results[['Nome Fantasia', 'Telefone', 'Tempo Sem Backup']].to_string())